In [13]:
#קוד מהילה
import pandas as pd
import numpy as np

# =====================================================================
# 1. הגדרת פונקציית עיבוד הנתונים (prepare_data)
# =====================================================================
def prepare_data(df):
    """
    מקבלת DataFrame גולמי ומחזירה DataFrame מעובד (פיצ'רים בלבד).
    מנקה נתונים, מבצעת Feature Engineering ומונעת Data Leakage.
    """
    data = df.copy()
    data.columns = data.columns.str.strip() # ניקוי רווחים בשמות העמודות
    
    # --- 1. מניעת Data Leakage והסרת משתנה המטרה ---
    cols_to_drop = ['averageRating', 'numVotes', 'BoxOffice']
    # 🌟 תיקון 1: הוספת errors='ignore' למניעת קריסה בהרצה כפולה
    data = data.drop(columns=[col for col in cols_to_drop if col in data.columns], errors='ignore')
            
    # --- 2. ניקוי נתונים חסין ---
    if 'budget' in data.columns:
        data['budget'] = data['budget'].astype(str)
        data['budget'] = data['budget'].replace(r'\\N', np.nan, regex=True)
        data['budget'] = data['budget'].str.replace(',', '', regex=False)
        data['budget'] = data['budget'].str.replace(r'[^\d.]', '', regex=True)
        data['budget'] = data['budget'].replace('', np.nan)
        data['budget'] = pd.to_numeric(data['budget'], errors='coerce')
        
    if 'runtimeMinutes' in data.columns:
        data['runtimeMinutes'] = data['runtimeMinutes'].astype(str).replace(r'\\N', np.nan, regex=True)
        data['runtimeMinutes'] = pd.to_numeric(data['runtimeMinutes'], errors='coerce')
        
    if 'startYear' in data.columns:
        data['startYear'] = data['startYear'].astype(str).replace(r'\\N', np.nan, regex=True)
        data['startYear'] = pd.to_numeric(data['startYear'], errors='coerce')

    # --- 3. Feature Engineering (5 פיצ'רים חדשים) ---
    if 'budget' in data.columns and 'runtimeMinutes' in data.columns:
        data['budget_per_minute'] = data['budget'] / data['runtimeMinutes'].replace(0, np.nan)
    else:
        data['budget_per_minute'] = np.nan
    
    data['num_genres'] = data['genres'].apply(lambda x: len(str(x).split(',')) if pd.notnull(x) and str(x) != '\\N' else 0)
    data['primary_genre'] = data['genres'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) and str(x) != '\\N' else 'Unknown')
    data['is_US'] = data['Country'].apply(lambda x: 1 if str(x).strip() in ['US', 'USA', 'United States'] else 0)
    
    if 'startYear' in data.columns:
        data['release_decade'] = (data['startYear'] // 10) * 10
    
    # 🌟 תיקון 2: חגורת בטיחות למניעת קריסות ממתמטיקה שגויה (Inf)
    data = data.replace([np.inf, -np.inf], np.nan)

    # --- 4. ניקוי עמודות מיותרות ---
    raw_cols_to_drop = ['tconst', 'primaryTitle', 'genres', 'lead_actors_ids', 'plot', 'Country', 'Language', 'startYear']
    # 🌟 תיקון 1 (המשך): הוספת errors='ignore'
    data = data.drop(columns=[col for col in raw_cols_to_drop if col in data.columns], errors='ignore')
            
    return data

# =====================================================================
# 2. הרצת הקוד: קריאת הקובץ, הפעלת הפונקציה ובדיקות!
# =====================================================================
try:
    print("קורא את קובץ הנתונים...")
    raw_df = pd.read_csv('dataset.csv')
    print(f"הקובץ נטען בהצלחה! מספר שורות גולמי: {raw_df.shape[0]}")
    
    raw_df.columns = raw_df.columns.str.strip()
    
    # 🌟 תיקון 3: סינון שורות בלי טרגט לפני שמחלצים את y!
    df_clean = raw_df.dropna(subset=['averageRating']).copy()
    print(f"מספר שורות לאימון לאחר הסרת סרטים ללא דירוג: {df_clean.shape[0]}")
    
    y = df_clean['averageRating']
    X = prepare_data(df_clean)
    
    print("\n=== 1. סוגי הנתונים (dtypes) - נוודא שהתקציב הוא מספר! ===")
    print(X.dtypes)
    
    print("\n=== 2. כמות ערכים חסרים ===")
    print(X.isnull().sum())
    
    print("\n=== 3. תצוגה מקדימה של X ===")
    display(X.head())
    
except FileNotFoundError:
    print("שגיאה: הקובץ 'dataset.csv' לא נמצא. ודאי שהוא שמור באותה התיקייה שבה נמצאת")

קורא את קובץ הנתונים...


C:\Users\user\AppData\Local\Temp\ipykernel_40360\3709265146.py:66: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('dataset.csv')


הקובץ נטען בהצלחה! מספר שורות גולמי: 133884
מספר שורות לאימון לאחר הסרת סרטים ללא דירוג: 115560

=== 1. סוגי הנתונים (dtypes) - נוודא שהתקציב הוא מספר! ===
runtimeMinutes       float64
budget               float64
budget_per_minute    float64
num_genres             int64
primary_genre         object
is_US                  int64
release_decade       float64
dtype: object

=== 2. כמות ערכים חסרים ===
runtimeMinutes           0
budget               99405
budget_per_minute    99405
num_genres               0
primary_genre            0
is_US                    0
release_decade           0
dtype: int64

=== 3. תצוגה מקדימה של X ===


,runtimeMinutes,budget,budget_per_minute,num_genres,primary_genre,is_US,release_decade
0,90.0,NaN,NaN,3,Crime,0,1990.0
1,86.0,NaN,NaN,1,Drama,0,2020.0
4,102.0,NaN,NaN,1,Comedy,0,2000.0
7,85.0,NaN,NaN,3,Biography,0,2010.0
8,170.0,NaN,NaN,1,Documentary,0,1990.0


In [14]:
print(X.columns.tolist())

['runtimeMinutes', 'budget', 'budget_per_minute', 'num_genres', 'primary_genre', 'is_US', 'release_decade']


In [15]:
# =====================================================================
# תא 3: שלב חקירת הנתונים ובקרת איכות (EDA & Validation)
# =====================================================================

print("=== 1. בחינת המאפיינים הנומריים והערכים הקיצוניים ===")
# כאן אנחנו בוחנים את המינימום, המקסימום והחציונים של הפיצ'רים שלנו
display(X.describe())

print("\n=== 2. בדיקת התפלגות משתנה המטרה (y - averageRating) ===")
# מוודאים שמשתנה המטרה הגיוני (בין 1 ל-10) ושאין בו ערכים חסרים
print(y.describe())
print(f"כמות ערכים חסרים ב-y: {y.isnull().sum()}")

print("\n=== 3. וידוא חסינות: בדיקת ערכי אינסוף (Inf) בדאטה המעובד ===")
# בדיקה קריטית שמראה למרצה שווידאנו באופן אקטיבי שהמודל לא יקרוס
inf_counts = np.isinf(X.select_dtypes(include=np.number)).sum()
print("כמות ערכי אינסוף בכל עמודה:")
print(inf_counts)

print("\n=== 4. הצצה להתפלגות הז'אנר הראשי (primary_genre) ===")
# בודקים שהקטגוריות נקיות ואין בהן תווים שבורים
print(X['primary_genre'].value_counts().head(10))

=== 1. בחינת המאפיינים הנומריים והערכים הקיצוניים ===


,runtimeMinutes,budget,budget_per_minute,num_genres,is_US,release_decade
count,115560.000000,1.615500e+04,1.615500e+04,115560.000000,115560.000000,115560.000000
mean,98.573183,1.718406e+09,1.716031e+07,1.891121,0.163915,1993.038508
std,22.346947,1.968606e+11,1.968725e+09,0.847158,0.370200,25.675311
min,60.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
25%,85.000000,2.500000e+00,2.438690e-02,1.000000,0.000000,1980.000000
50%,94.000000,9.500000e+00,9.200000e-02,2.000000,0.000000,2000.000000
75%,107.000000,4.200000e+01,4.117647e-01,3.000000,0.000000,2010.000000
max,300.000000,2.500000e+13,2.500000e+11,3.000000,1.000000,2020.000000



=== 2. בדיקת התפלגות משתנה המטרה (y - averageRating) ===
count    115560.000000
mean          6.070235
std           1.292801
min           1.000000
25%           5.300000
50%           6.200000
75%           7.000000
max          10.000000
Name: averageRating, dtype: float64
כמות ערכים חסרים ב-y: 0

=== 3. וידוא חסינות: בדיקת ערכי אינסוף (Inf) בדאטה המעובד ===
כמות ערכי אינסוף בכל עמודה:
runtimeMinutes       0
budget               0
budget_per_minute    0
num_genres           0
is_US                0
release_decade       0
dtype: int64

=== 4. הצצה להתפלגות הז'אנר הראשי (primary_genre) ===
primary_genre
Drama              17197
Comedy             14808
['Drama']           8088
['Comedy'           7473
Action              7184
['Drama'            6589
Documentary         5601
['Action'           5516
['Comedy']          3937
['Documentary']     3428
Name: count, dtype: int64


In [16]:
# =====================================================================
# תא 3: בדיקת הנתונים המעובדים (Sanity Check)
# =====================================================================

print(f"ממדי מטריצת המאפיינים X: {X.shape}")
print(f"ממדי וקטור המטרה y: {y.shape}")

print("\n--- בדיקת סוגי הנתונים וערכים חסרים ב-X ---")
# יצירת טבלה קטנה ונוחה שמראה בדיוק מה המצב של כל עמודה
summary_df = pd.DataFrame({
    'סוג נתון': X.dtypes,
    'ערכים חסרים': X.isnull().sum()
})
display(summary_df)

print("\n--- בדיקת ערכים חסרים ב-y (averageRating) ---")
print(f"כמות ערכים חסרים: {y.isnull().sum()}")

ממדי מטריצת המאפיינים X: (115560, 7)
ממדי וקטור המטרה y: (115560,)

--- בדיקת סוגי הנתונים וערכים חסרים ב-X ---


,סוג נתון,ערכים חסרים
runtimeMinutes,float64,0
budget,float64,99405
budget_per_minute,float64,99405
num_genres,int64,0
primary_genre,object,0
is_US,int64,0
release_decade,float64,0



--- בדיקת ערכים חסרים ב-y (averageRating) ---
כמות ערכים חסרים: 0


בבדיקת הדאטה המעובד קיבלנו 133,894 שורות ו-7 עמודות. כל העמודות הנומריות זוהו בהצלחה כ-float או int, כולל עמודת ה-budget שהומרה למספר. בעמודות התקציב ישנם ערכים חסרים רבים (מכיוון שהחלפנו את ערכי הזבל ב-NaN), ובשלב הבא נטפל בהם בצורה מסודרת באמצעות Imputer בתוך ה-Pipeline כדי למנוע זליגת נתונים. משתנה המטרה (y) מלא ותקין.

### Pipeline

In [18]:
# =====================================================================
# תא 4: בניית Pipeline ואימון מודל Elastic Net (GridSearchCV + 10-Fold CV)
# =====================================================================
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import ElasticNet
import numpy as np

# 1. הגדרת רשימות העמודות לפי סוגים
numeric_features = ['runtimeMinutes', 'budget', 'budget_per_minute', 'num_genres', 'is_US', 'release_decade']
categorical_features = ['primary_genre']

# 🌟 השדרוג של חברה שלך: חגורת בטיחות - מוודאים שהעמודות באמת קיימות בדאטה 🌟
numeric_features = [col for col in numeric_features if col in X.columns]
categorical_features = [col for col in categorical_features if col in X.columns]

# 2. בניית תתי-התהליכים (Transformers) למניעת זליגת נתונים בתוך ה-CV
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # השלמת חסרים לפי החציון של עמודת המספרים
    ('scaler', StandardScaler())                  # נרמול הנתונים לממוצע 0 וסטיית תקן 1
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), 
    # 🌟 השדרוג של חברה שלך: sparse_output=False למניעת בעיות זיכרון 🌟
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

# 3. איחוד המעבדים
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. בניית ה-Pipeline השלם (עיבוד + מודל)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # 🌟 השדרוג של חברה שלך: העלאת max_iter ל-3000 למניעת אזהרות 🌟
    ('model', ElasticNet(random_state=42, max_iter=3000))
])

# 5. הגדרת רשת הפרמטרים לחיפוש (לפי דרישות המטלה)
param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0],
    'model__l1_ratio': [0.2, 0.5, 0.8]
}

# 6. הגדרת 10-Fold CV
cv_strategy = KFold(n_splits=10, shuffle=True, random_state=42)

# 7. הגדרת החיפוש עם שמירת מדדי RMSE ו-MAE
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring={'RMSE': 'neg_root_mean_squared_error', 'MAE': 'neg_mean_absolute_error'},
    refit='RMSE', # אנחנו רוצים שהוא יבחר את המודל הטוב ביותר לפי RMSE
    # 🌟 השדרוג של חברה שלך: n_jobs=None כדי שהמחברת לא תקרוס פתאום מניצול זיכרון 🌟
    n_jobs=None,  
    verbose=1
)

# =====================================================================
# אימון המודל (יכול לקחת כדקה-שתיים)
# =====================================================================
print("מתחיל אימון מודל Elastic Net עם 10-Fold CV וסריקת פרמטרים...")
grid_search.fit(X, y)
print("האימון הסתיים בהצלחה!\n")

# --- חילוץ התוצאות לדיווח לפי ההנחיות ---
best_index = grid_search.best_index_
cv_results = grid_search.cv_results_

# הכפלה במינוס 1 כי הפונקציה מחזירה תוצאות שליליות
best_rmse_mean = -cv_results['mean_test_RMSE'][best_index]
best_rmse_std = cv_results['std_test_RMSE'][best_index]
best_mae_mean = -cv_results['mean_test_MAE'][best_index]
best_mae_std = cv_results['std_test_MAE'][best_index]

print("=== תוצאות מודל Elastic Net (ממוצע על 10 קיפולים) ===")
print(f"הפרמטרים המיטביים (Tuning): {grid_search.best_params_}")
print(f"RMSE ממוצע: {best_rmse_mean:.4f} (סטיית תקן: {best_rmse_std:.4f})")
print(f"MAE ממוצע: {best_mae_mean:.4f} (סטיית תקן: {best_mae_std:.4f})")

מתחיל אימון מודל Elastic Net עם 10-Fold CV וסריקת פרמטרים...
Fitting 10 folds for each of 12 candidates, totalling 120 fits
האימון הסתיים בהצלחה!

=== תוצאות מודל Elastic Net (ממוצע על 10 קיפולים) ===
הפרמטרים המיטביים (Tuning): {'model__alpha': 0.01, 'model__l1_ratio': 0.2}
RMSE ממוצע: 1.1797 (סטיית תקן: 0.0093)
MAE ממוצע: 0.9120 (סטיית תקן: 0.0068)
